# Corporate Bankruptcy Prediction — step-by-step analysis
UCI Polish Companies Bankruptcy (id=365). Runs the same functions as `src/train.py`. Outputs appear when you run the cells (nothing is pre-computed).

In [ ]:
import sys
sys.path.insert(0, '../src')
%matplotlib inline
from utils import *
from data_loader import load_data, clean_data, inspect_dataset
from eda import run_eda
from preprocessing import analyze_imbalance
from train import get_model_specs, split_data, make_cv, tune_all, save_final_model, save_sample_input
from evaluate import evaluate_models, build_comparison, interpret_models
set_seed(); ensure_dirs()

## Step 1 — Load directly from UCI and inspect

In [ ]:
X, y, info = load_data()
report = inspect_dataset(X, y, info)

## Data cleaning and Step 2 — EDA

In [ ]:
X, y, n_dup = clean_data(X, y)
run_eda(X, y)

## Step 3 — Class imbalance

In [ ]:
imbalance = analyze_imbalance(y)

## Step 4 — Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

## Steps 5–7 — Pipelines and tuning (training data only; may take several minutes)

In [ ]:
searches = tune_all(get_model_specs(), X_train, y_train, make_cv(CV_FOLDS), N_ITER)
models = {n: s.best_estimator_ for n, s in searches.items()}

## Steps 8–9 — Evaluation on the untouched test set and comparison

In [ ]:
test_df, scores = evaluate_models(models, X_test, y_test)
comparison = build_comparison(test_df, searches)
comparison.round(4)

## Step 10 — Interpretation

In [ ]:
best_name = comparison.index[0]
interpret_models(models, best_name, X_test, y_test)

## Steps 11–12 — Save final model and predict

In [ ]:
save_final_model(best_name, models, searches, comparison, X_train)
save_sample_input(X_test, y_test)
from predict import load_artifacts, predict_frame, format_prediction
model, meta = load_artifacts()
pred, proba = predict_frame(X_test.head(3), model, meta)
for i, p in enumerate(pred):
    print(format_prediction(p, None if proba is None else proba[i], meta, row=i))